# 3.1 Mixture of experts: many parameters stored, few used per image

At the end of this notebook you can:

- say why a mixture of experts is compared with a dense head that **uses** as many
  parameters per image, and point at that control in an experiment;
- tell the parameters a sparse mixture of experts **stores** from the parameters it
  **uses** for one image, and say when using fewer makes it faster;
- say why it wins here: the router learns to split the classes over the experts, so each
  expert separates only part of them; with the router frozen at its start, the gain is gone.

**The situation.** A service labels a large stream of photos of four kinds: textures (DTD),
satellite tiles (EuroSAT), clothing (Fashion-MNIST) and flowers (Oxford Flowers), 169 classes
in all. It works through the stream in batches with a fixed budget of arithmetic per image,
a few thousand parameters used per image, so one small dense head has to serve four domains
with the same few parameters. Such a budget is real when there is very much data: large
models are trained and served on huge amounts of it, and every image or token pays the
arithmetic of every parameter it passes through; here the budget simulates that on a laptop.
A **mixture of experts** (MoE) answers it: it stores many experts and runs only one of them per image. The
slides of lesson 3 drew it as the third route after the gate and the highway; this notebook
builds it.

## 3.1.1 The mix, summarised in 64 numbers

The four `dinov2-small` caches on the hub hold a 384-dim vector per image: `dinov2-small` is
a pretrained image encoder, as in lesson 1, that turns a photo into 384 numbers. `load_mix`
(in `mixture.py`, next to this notebook) stacks them into one set, shifts each domain's labels past the classes before it, so that class 0 of EuroSAT does
not collide with class 0 of DTD, and keeps the domain of every row. The models never see the
domain; the notebook keeps it to read accuracy and routing per domain afterwards.

The 64-number summary is PCA (lesson 1, `01` §1.1): the 64 directions along which the train
vectors vary most, fitted on the train rows only, and every vector projected onto them. Both
models below see the same 64 numbers. The summary is artificial: it keeps the router (the part of a mixture of experts that picks
an expert, 3.1.2) cheap.
On the full 384 numbers the router alone would be `384 · 8 + 8 = 3,080` parameters, more than
the smallest mixture of experts below uses per image in all; on 64 numbers it is 520, small next to the
experts, as a router is in large networks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from mixture import DOMAINS, load_mix, time_per_image, train
from notebooktester import param
from torch import nn
from vectormesh.components import SparseMoE

full_train = load_mix("train")  # 384-dim vectors, labels, domain of every row
full_test = load_mix("test")
K = int(full_train[1].max()) + 1

P = 64
center = full_train[0].mean(0)
_, _, Vh = torch.linalg.svd(full_train[0] - center, full_matrices=False)
V = Vh[:P].T  # (384, P): the 64 directions of largest variance
train_rows = ((full_train[0] - center) @ V, full_train[1], full_train[2])
test_rows = ((full_test[0] - center) @ V, full_test[1], full_test[2])

kept = (train_rows[0] ** 2).sum() / ((full_train[0] - center) ** 2).sum()
print(f"{K} classes; {P} numbers per image keep {kept:.0%} of the variance of the 384")
print(f"{'domain':15s} {'classes':>7s} {'train':>6s} {'test':>5s}")
for i, name in enumerate(DOMAINS):
    classes = len(np.unique(train_rows[1][train_rows[2] == i]))
    n_train, n_test = (train_rows[2] == i).sum(), (test_rows[2] == i).sum()
    print(f"{name:15s} {classes:7d} {n_train:6d} {n_test:5d}")
print(f"{'all':15s} {K:7d} {len(train_rows[1]):6d} {len(test_rows[1]):5d}")

## 3.1.2 Two heads, counted

**The dense head** is lesson 1's head with one hidden layer of width `w`:
`(B, 64)` → `Linear(64, w)` → GELU → `Linear(w, 169)` → `(B, 169)`, one score per class.
It has `64·w + w` parameters in the first layer and `w·169 + 169` in the second, and every
image passes through all of them.

**The sparse mixture of experts** (sparse: only some experts run per input) holds 8 experts,
each a dense head of width `w`, and a **router**: a `Linear(64, 8)` that gives every image 8
scores, turned into 8 probabilities by a softmax. Only the expert with the highest
probability runs on the image (**top-1**), and its 169 scores are multiplied by that
probability. Choosing the top expert is an argmax, which has no gradient; the probability
that multiplies the chosen expert's output has one, and through it the router learns which
expert to choose. In code it is `SparseMoE(experts, 64, 169, k=1)`: the experts, the input
size, the output size and `k`, the number of experts that run per image.

<svg width="720" height="300" viewBox="0 0 720 300" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="300" fill="white"/>
<defs><marker id="m1" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="22" font-weight="bold" fill="#222">dense head</text>
<text x="10" y="58" fill="#222">image</text>
<line x1="52" y1="54" x2="158" y2="54" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="105" y="46" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<rect x="160" y="32" width="250" height="44" rx="8" fill="#cfe3f3" stroke="#555"/>
<text x="285" y="52" text-anchor="middle" fill="#222">Linear(64, w), GELU, Linear(w, 169)</text>
<text x="285" y="68" text-anchor="middle" font-size="11" fill="#222">every parameter used for every image</text>
<line x1="410" y1="54" x2="578" y2="54" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="494" y="46" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 169)</text>
<text x="584" y="58" fill="#222">scores</text>
<text x="10" y="112" font-weight="bold" fill="#222">sparse mixture of experts, top-1</text>
<text x="10" y="208" fill="#222">image</text>
<line x1="52" y1="204" x2="110" y2="204" stroke="#444" stroke-width="1.5"/>
<path d="M 110 204 L 110 150 L 158 150" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 110 204 L 110 240 L 298 240" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="52" y="222" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<rect x="160" y="128" width="150" height="44" rx="8" fill="#f6d9a8" stroke="#555"/>
<text x="235" y="148" text-anchor="middle" fill="#222">router</text>
<text x="235" y="164" text-anchor="middle" font-size="11" fill="#222">Linear(64, 8), softmax</text>
<line x1="310" y1="150" x2="388" y2="150" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="349" y="142" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 8)</text>
<text x="394" y="146" fill="#222">top-1: expert e,</text>
<text x="394" y="162" fill="#222">probability p</text>
<path d="M 450 168 L 450 196" fill="none" stroke="#444" stroke-width="1.5" stroke-dasharray="4 3" marker-end="url(#m1)"/>
<text x="458" y="188" font-size="11" fill="#222">picks the expert</text>
<rect x="300" y="200" width="300" height="80" rx="8" fill="none" stroke="#999" stroke-dasharray="4 3"/>
<rect x="310" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="346" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="382" y="208" width="30" height="64" rx="4" fill="#cfe3f3" stroke="#555"/>
<rect x="418" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="454" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="490" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="526" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="562" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<text x="397" y="244" text-anchor="middle" font-size="11" fill="#222">e</text>
<text x="450" y="294" text-anchor="middle" font-size="11" fill="#222">8 experts, each a dense head of width w; only e runs on this image</text>
<line x1="600" y1="240" x2="638" y2="240" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="644" y="236" fill="#222">× p</text>
<text x="644" y="252" font-family="monospace" font-size="11" fill="#222">(B, 169)</text>
</svg>


The shapes, arrow by arrow:

- `(B, 64)`: a batch of `B` images, 64 numbers each, goes to the router and to the experts.
- `(B, 8)`: the router's 8 probabilities per image; the largest names the expert `e` that
  runs, and its value is the `p` that scales that expert's output.
- `(B, 169)`: the rows sent to expert `e` are gathered into one smaller batch, the expert
  gives 169 scores per row, and the scores, times `p`, are scattered back to their place in
  the batch.

So the MoE has two counts. **Stored**: the router and all 8 experts, everything that is
trained and kept. **Used** for one image: the router and one expert; the other 7 experts
take no part in that image. A used parameter is one multiply and one add per image, so the
used count measures arithmetic, not time; 3.1.6 asks whether the clock agrees.

Why experts could help under a tight budget: a dense head of width 6 has a hidden layer of 6
numbers to separate all 169 classes; an expert only has to separate the classes the router
sends it.

**The control.** The budget is on the parameters used per image. A MoE that beats a
smaller dense head could win only because it uses more, so each MoE is compared with the
dense head whose used parameters come closest to its own, router included. That pair
differs in one factor: what the MoE stores beyond what it uses, seven more experts to choose
from. A comparison that looks just as natural is the dense head as wide as one expert, "one
expert alone". It leaves out the router's parameters, so it flatters the MoE; 3.1.4 shows by
how much.

In [ ]:
N_EXPERTS = 8
WIDTHS = [4, 8, 16]  # the width of one expert


def dense(width: int, dim: int = P) -> nn.Module:
    return nn.Sequential(nn.Linear(dim, width), nn.GELU(), nn.Linear(width, K))


def moe(width: int, dim: int = P) -> nn.Module:
    return SparseMoE([dense(width, dim) for _ in range(N_EXPERTS)], dim, K, k=1)


def stored(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def used(model: nn.Module) -> int:
    # The parameters one image passes through: router and one expert for a MoE.
    if isinstance(model, SparseMoE):
        return stored(model.router) + stored(model.experts[0])
    return stored(model)


def control_width(width: int, dim: int = P) -> int:
    # A dense head of width d has d * (dim + 1 + K) + K parameters; take the d closest
    # to the used parameters of moe(width, dim).
    return round((used(moe(width, dim)) - K) / (dim + 1 + K))


CONTROL = {w: control_width(w) for w in WIDTHS}
DENSE_WIDTHS = sorted({24, *WIDTHS, *CONTROL.values()})

print(f"router: {stored(moe(4).router)} parameters\n")
print(f"{'model':>10s} {'stored':>8s} {'used per image':>15s}")
for w in WIDTHS:
    for model, name in [
        (moe(w), f"MoE 8 x {w}"),
        (dense(CONTROL[w]), f"dense {CONTROL[w]}"),
    ]:
        print(f"{name:>10s} {stored(model):8,d} {used(model):15,d}")
print(f"\ndense widths trained: {DENSE_WIDTHS}")

Each MoE stores 5.8 to 7.2 times what it uses. The router is `64 · 8 + 8 = 520` parameters, a
fifth of what the MoE `8 × 8` uses. Width 24 is trained as the top of the dense curve.

## 3.1.3 The experiment

**Question.** At the same parameters used per image, does the MoE classify the mix better
than a dense head?

**Setup.** The dense heads of every width and the three MoEs, one seed each, all trained the
same way for 100 epochs (`train` in `mixture.py`, whose docstring gives the recipe); the epoch
with the best validation accuracy is kept and the test split is scored once. The run records
the test accuracy, the expert the router picks for every test image, and the time per image
that 3.1.6 reads.

**Expect.** Write down: will the MoEs lie above the dense curve, at the same used parameters?
And how far above the dense head as wide as one of their experts?

In [ ]:
SEED = 0
EPOCHS = param(100, test=1)
CALLS = param(200, test=5)  # timed forward passes per batch size

X_test, y_test, domain_test = test_rows


def run(name: str, make) -> dict:
    model, X_std, pred, valid_curve = train(make, train_rows, X_test, SEED, EPOCHS)
    right = (pred == y_test).numpy()
    with torch.no_grad():
        route = (
            model.router(X_std).argmax(1).numpy()
            if isinstance(model, SparseMoE)
            else None
        )
    return {
        "name": name,
        "used": used(model),
        "test": right.mean(),
        "valid_curve": valid_curve,
        "route": route,
        "X_std": X_std,
        "us_batch1": time_per_image(model, 1, P, CALLS),
        "us_batch256": time_per_image(model, 256, P, CALLS),
    }


dense_runs = {d: run(f"dense {d}", lambda d=d: dense(d)) for d in DENSE_WIDTHS}
moe_runs = {w: run(f"MoE 8 x {w}", lambda w=w: moe(w)) for w in WIDTHS}

print(f"{'model':>10s} {'used':>6s} {'kept epoch':>10s} {'test accuracy':>14s}")
for r in [*dense_runs.values(), *moe_runs.values()]:
    kept = int(np.argmax(r["valid_curve"])) + 1
    print(f"{r['name']:>10s} {r['used']:6,d} {kept:10d} {r['test']:14.3f}")

**Has every run settled?** A model scored while it was still learning would measure its
budget of epochs, not its architecture. The validation accuracy per epoch shows where each run
stands at the end; the dot marks the kept epoch.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
epochs = np.arange(1, EPOCHS + 1)
for a, runs, title in [
    (axs[0], dense_runs, "dense heads"),
    (axs[1], moe_runs, "MoE, 8 experts"),
]:
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(runs)))
    for r, color in zip(runs.values(), colors):
        best = int(np.argmax(r["valid_curve"]))
        a.plot(epochs, r["valid_curve"], color=color, lw=1.2, label=r["name"])
        a.plot(best + 1, r["valid_curve"][best], "o", color=color)
    a.set_title(title, fontsize=10)
    a.set_xlabel("epoch")
    a.grid(alpha=0.3)
    a.legend(frameon=False, fontsize=7, ncol=2, loc="lower right")
axs[0].set_ylabel("validation accuracy")
fig.suptitle(
    "The narrow dense heads still creep up at epoch 100; the MoEs have flattened"
)
plt.tight_layout()
plt.show()

**Reading.** The MoEs and the dense heads from width 10 on are flat over the last 20 epochs.
The narrow dense heads 4, 6 and 8 still creep up, by under a point from epoch 80 to 100: far
less than the gaps 3.1.4 reads at their budgets.

## 3.1.4 The MoE against the dense curve

The test accuracy of every run against the parameters it uses per image: the dense heads as
one curve, the MoEs as points.

In [ ]:
fig, a = plt.subplots(figsize=(7.5, 4.2))
xd = [r["used"] for r in dense_runs.values()]
yd = [r["test"] for r in dense_runs.values()]
a.plot(xd, yd, "o-", color="#0072B2", label="dense head")
for d, x, y in zip(dense_runs, xd, yd):
    a.annotate(
        str(d),
        (x, y),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=8,
        color="#0072B2",
    )
for w, r in moe_runs.items():
    a.plot(
        r["used"],
        r["test"],
        "s",
        color="#D55E00",
        ms=8,
        label="MoE, 8 experts, top-1" if w == WIDTHS[0] else None,
    )
    a.annotate(
        f"8 x {w}",
        (r["used"], r["test"]),
        textcoords="offset points",
        xytext=(8, 0),
        ha="left",
        va="center",
        fontsize=8,
        color="#D55E00",
    )
a.set_xscale("log")
a.set_xticks(xd, [f"{x:,}" for x in xd], fontsize=7, rotation=45)
a.minorticks_off()
a.set_ylim(min(yd) - 0.05, max(yd) + 0.05)
a.set_xlabel("parameters used per image (log); dense width above each point")
a.set_ylabel("test accuracy, 4 domains")
a.set_title(
    "Below 4,000 used the MoEs lie above the dense curve; at 4,433 it has caught up"
)
a.grid(alpha=0.3)
a.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

print(
    f"{'MoE':>9s} {'used':>6s} {'test':>6s} | {'control: same used':>24s} | {'one expert alone':>24s}"
)
for w, r in moe_runs.items():
    one, same = dense_runs[w], dense_runs[CONTROL[w]]
    print(
        f"{r['name']:>9s} {r['used']:6,d} {r['test']:6.3f} | "
        f"{same['name']:>9s} {same['used']:5,d} {same['test']:.3f} {100 * (r['test'] - same['test']):+5.1f} | "
        f"{one['name']:>9s} {one['used']:5,d} {one['test']:.3f} {100 * (r['test'] - one['test']):+5.1f}"
    )

**Reading.** At the two tight budgets the MoE lies above the dense curve. Against its
control, the dense head with the same used parameters, the MoE `8 × 4` gains 12.0 points
(0.712 against 0.592 for dense 6) and the MoE `8 × 8` 5.2 points (0.813 against 0.760 for
dense 10). At 4,433 used the dense head has caught up: the MoE `8 × 16` reaches 0.829 against
0.840 for dense 18 (−1.1), and each of its experts learns from about 2,000 images, the data
per expert of the slides. The dense curve flattens from width 16 on (0.827, 0.840, 0.855):
once the budget is ample, compute is no longer the bottleneck and the experts have little to
add.

"One expert alone" flatters the MoE: against dense 4 and dense 8 it gains 28.6 and 11.8
points, because those heads leave out the router's 520 parameters. The control is the head
with the same used parameters.

This is one seed; another seed could move each number by a point or two. That does not reach
the gains at 1,625 and 2,561 used; the −1.1 at 4,433 it could turn around.

## 3.1.5 What the router learned

**Question.** The router never saw which domain an image comes from. Does it still send each
domain to experts of its own?

**Setup.** For the MoE `8 × 8`: the share of each domain's test images that the router sends
to each expert (cells below 0.05 are left blank), and per class the share of its test images
that go to the one expert it is sent to most, averaged over the 169 classes; that share is 1
when all images of a class go to the same expert. The vectors of a class lie close together, so even an untrained router sends a class's
images partly to one expert; both are therefore also computed for the untrained router of the
same seed. Last, for each
expert, the number of classes whose images it gets most of.

**Expect.** Write down: does each domain get experts of its own, and does the untrained
router split the mix too?

In [ ]:
def shares(route: np.ndarray) -> np.ndarray:
    # Row i: the share of domain i's test images routed to each expert.
    return np.array(
        [
            np.bincount(route[domain_test == i], minlength=N_EXPERTS)
            / (domain_test == i).sum()
            for i in range(len(DOMAINS))
        ]
    )


trained = moe_runs[8]
torch.manual_seed(SEED)
with torch.no_grad():
    untrained_route = moe(8).router(trained["X_std"]).argmax(1).numpy()

fig, axs = plt.subplots(1, 2, figsize=(13, 3.4))
for a, (title, route) in zip(
    axs, [("untrained router", untrained_route), ("trained router", trained["route"])]
):
    table = shares(route)
    a.imshow(table, cmap="Greys", vmin=0, vmax=1)
    for i in range(len(DOMAINS)):
        for e in range(N_EXPERTS):
            if table[i, e] >= 0.05:
                a.text(
                    e,
                    i,
                    f"{table[i, e]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if table[i, e] > 0.5 else "black",
                )
    a.set_yticks(range(len(DOMAINS)), DOMAINS)
    a.set_xticks(range(N_EXPERTS))
    a.set_xlabel("expert")
    a.set_title(f"MoE 8 x 8: {title}", fontsize=10)
fig.suptitle("Share of each domain's test images sent to each expert")
plt.tight_layout()
plt.show()


def class_share(route: np.ndarray) -> float:
    # Per class, the share of its test images sent to its most used expert; mean over classes.
    y = y_test.numpy()
    return float(
        np.mean(
            [
                np.bincount(route[y == k], minlength=N_EXPERTS).max() / (y == k).sum()
                for k in np.unique(y)
            ]
        )
    )


def classes_per_expert(route: np.ndarray) -> np.ndarray:
    # For each expert, the number of classes whose test images it gets most of.
    y = y_test.numpy()
    home = [
        np.bincount(route[y == k], minlength=N_EXPERTS).argmax() for k in np.unique(y)
    ]
    return np.bincount(home, minlength=N_EXPERTS)


print(
    f"share of a class's images at its most used expert: untrained {class_share(untrained_route):.2f}, "
    f"trained {class_share(trained['route']):.2f}"
)
y = y_test.numpy()
print("per domain, untrained -> trained:")
for i, name in enumerate(DOMAINS):
    mask = domain_test == i
    shares_d = [
        np.mean(
            [
                np.bincount(r[mask & (y == k)], minlength=N_EXPERTS).max()
                / (mask & (y == k)).sum()
                for k in np.unique(y[mask])
            ]
        )
        for r in (untrained_route, trained["route"])
    ]
    print(f"  {name:15s} {shares_d[0]:.2f} -> {shares_d[1]:.2f}")
counts = classes_per_expert(trained["route"])
print(
    f"classes per expert (most of their images): {' '.join(map(str, counts))} (sum {counts.sum()})"
)

**Reading.** The vectors are not spread evenly, so even the untrained router sends a class's
images to its most used expert 59% of the time. Training strengthens that split to 85%,
most where the untrained router split least: EuroSAT goes from 0.45 to 0.91 and
Fashion-MNIST from 0.42 to 0.88; Flowers starts at 0.68 and ends at 0.89.
EuroSAT and DTD get experts of their own: EuroSAT goes mostly to expert 0 (71%), DTD mostly to
experts 2 and 4. Fashion-MNIST and Flowers share experts. Each expert gets most of the images
of 13 to 33 classes, at most a fifth of the 169, where the control's hidden layer has to
separate all 169.

The router is never given the domain, nor which classes go to which expert. The slides showed
the opposite outcome, router collapse, every image to one expert; here every expert gets
classes of its own.


**Question.** The untrained router already splits the mix in part. Does the gain of 3.1.4
come from the split the router learns, or would the experts win behind any fixed split?

**Setup.** The MoEs `8 × 4` and `8 × 8` once more, with the router frozen at its random start:
its parameters get no gradient, so it keeps the split of the untrained router above, and
only the experts train. Same recipe and seed. Each is set beside the MoE with a trained router
and beside its control from 3.1.4.

**Expect.** Write down: with a frozen router, does the MoE still beat its control?


In [ ]:
def frozen_moe(width: int) -> nn.Module:
    # The router keeps its random start: no gradient reaches it, only the experts train.
    model = moe(width)
    for p in model.router.parameters():
        p.requires_grad_(False)
    return model


print(f"{'MoE':>9s} {'router trained':>15s} {'router frozen':>14s} | {'control':>17s}")
for w in WIDTHS[:2]:
    _, _, pred, _ = train(lambda w=w: frozen_moe(w), train_rows, X_test, SEED, EPOCHS)
    frozen_test = (pred == y_test).float().mean().item()
    same = dense_runs[CONTROL[w]]
    print(
        f"{moe_runs[w]['name']:>9s} {moe_runs[w]['test']:15.3f} {frozen_test:14.3f} | "
        f"{same['name']:>9s} {same['test']:.3f}"
    )

**Reading.** No. With its router frozen at the random start, the MoE loses to its control:
0.539 against 0.592 for dense 6, and 0.726 against 0.760 for dense 10. Seven more experts
behind a split nobody trained do not beat a dense head of the same used parameters; the gain
of 3.1.4 comes from the split the router learns. The gaps between a trained and a frozen
router, 17 and 9 points, are far larger than what another seed moves.

That holds for experts this small: 4 to 8 units each, and each sees about 2,000 of the
16,000 training images, so a random split hands every expert a mixed slice of every class.
In large language models a fixed split by a hash of the token comes close to a learned
router (Roller et al., 2021, *Hash Layers for Large Sparse Models*); routing does not always
have to be learned.


## 3.1.6 Fewer used parameters, faster?

**Question.** A MoE and its control use as many parameters per image, so as much arithmetic.
Does the MoE answer as fast?

**Setup.** The runs of 3.1.3 timed every trained model on one CPU thread, on random vectors
of the same shape, at two batch sizes: batch 1, one photo per call, as an app that answers one
request while the user waits, and batch 256, a batch of the stream. Vectormesh checks tensor
shapes on every call while you design a model; the timing turns that check off, as a deployed
model runs without it.

**Expect.** Write down: is a MoE that runs one small expert slower or faster than one dense
head of the same used parameters?

In [ ]:
print(f"{'model':>10s} {'used':>6s} {'µs per image, batch 1':>22s} {'batch 256':>10s}")
for w in WIDTHS:
    for r in (dense_runs[CONTROL[w]], moe_runs[w]):
        print(
            f"{r['name']:>10s} {r['used']:6,d} {r['us_batch1']:22.1f} {r['us_batch256']:10.2f}"
        )

**Reading.** No: at batch 1 every MoE takes 26 to 29 µs per image against under 4 µs for its
control, about seven times as long, and at batch 256 0.59 to 0.64 µs against 0.07.
The MoE's time hardly moves with the width of its experts, so it is not their arithmetic but
the work around it: choosing the expert, sorting, gathering and scattering the rows, and a
separate matrix product for every expert that runs (one at batch 1, all eight at batch 256).
The dense head gets cheaper per image in a batch too, because one large matrix product over
many rows uses the processor better than many small ones.

**The crossover.** The cost of routing does not grow with the experts; the arithmetic an
expert saves does. So there is a crossover: below it, as with these small heads, a MoE is
slower than the dense head of the same used parameters; above it, with large experts in
batches, it is faster. The slides show that crossover. The very large models of the opening
sit far above it, where the router's price is a small share of the work, so the accuracy gain
at the same arithmetic of 3.1.4 becomes a gain in time there; that is why the budget here is
simulated. These are times on one CPU thread of this machine; another machine, or a GPU,
moves them.


## What you now know

- **The control** of a MoE is the dense head that uses as many parameters per image, router
  included; against it the MoE gained up to 12 points where the budget was tight, and the head
  as wide as one expert, without the router, overstates that gain.
- **Stored and used** are two counts: a MoE stores all its experts and uses the router and one
  of them per image, and fewer used parameters become speed only above a crossover, with large
  experts in batches.
- **The experts specialise**: the router is never given the domain; training strengthens the
  split the vectors already have (59% of a class's images at one expert untrained, 85%
  trained), so each expert separates only part of the 169 classes. With the router frozen at its
  random start the MoE loses to its control: the gain comes from the split it learns.

## Exercises

**E1: top-2.** Route every image to its two best experts
(`SparseMoE(experts, 64, 169, k=2, normalize=True)`; `normalize=True` rescales the two router
probabilities to sum to 1). Count the parameters it uses per image (the router and two
experts), find the dense head that uses as many, and train both at `w = 8`. Does a second
expert per image buy more than the same parameters as width?

**E2: more experts, and collapse.** The slides showed router collapse: every image sent to
one expert. Train MoEs of 32 and 64 experts of width 4, with and without the load-balancing
loss, in a copy of `train` from `mixture.py` whose loss line becomes
`loss = loss + 0.01 * load_balancing_loss(model)` (`load_balancing_loss` from
`vectormesh.components`). Count the experts that get at least one test image. What does the
router do with experts it does not need for four domains, and what does that cost in
accuracy?

**E3: the size of the summary.** Set `P` to 32 and to 128 and rerun 3.1.3 for the MoE `8 × 8`
and its control. How does the router's share of the used parameters change, and does the
gain at the same used parameters survive?

**E4: the control with as many stored parameters.** Train the dense head that **stores** as
much as the MoE `8 × 8`, which uses several times as many parameters per image. How much more
accurate is it, how much slower at batch 256, and which would you put in the service?